# 03 — Gazetteer (Pleiades + GeoNames)

Build a local gazetteer index used to resolve toponyms to geographic
coordinates and stable identifiers:

- **Pleiades** — a gazetteer of ancient places, downloaded once as a gzipped
  JSON dump and indexed by URI and by every known name form (primary,
  romanized, attested).
- **GeoNames** — a fallback web API used only for toponyms that Pleiades
  cannot resolve (see `04_disambiguation.ipynb`).

The Pleiades index is pickled to disk so it can be reloaded instantly by
`04_disambiguation.ipynb` without re-parsing the ~200 MB dump.

## CONFIG

In [ ]:
import gzip
import json
import logging
import pickle
import unicodedata
from pathlib import Path

import requests
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("aec_geoparser.gazetteer")

# Pleiades
PLEIADES_URL   = "https://atlantides.org/downloads/pleiades/json/pleiades-places-latest.json.gz"
PLEIADES_GZ    = Path("../data/cache/pleiades-places-latest.json.gz")
PLEIADES_INDEX = Path("../data/cache/pleiades_index.pkl")
FORCE_REBUILD  = False

# GeoNames (free account required at geonames.org)
GEONAMES_USER  = "your_geonames_username"
GEONAMES_URL   = "http://api.geonames.org/searchJSON"
GEONAMES_FCLS  = ["P", "H", "T", "L", "S"]   # feature classes
GEONAMES_ROWS  = 5                             # max candidates per query

USER_AGENT = "AeC-Geoparser/1.0 (github.com/gmancuso24/AeC_geoparser)"
TIMEOUT_S  = 30


## Normalization helper

Used to build lookup keys that are case- and diacritic-insensitive, so e.g. `"Caere"`, `"caere"` and accented variants collapse to the same key.

In [ ]:
def normalize(s: str) -> str:
    """Lowercase and strip diacritics to obtain a stable lookup key."""
    return unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()


## Download the Pleiades dump

Streamed with a progress bar; skipped if the file is already present.

In [ ]:
def download_pleiades(url: str, dest: Path) -> None:
    """Stream-download the gzipped Pleiades dump with a progress bar, if not already cached."""
    if dest.exists():
        logger.info("Pleiades dump already present at %s", dest)
        return

    dest.parent.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": USER_AGENT}
    with requests.get(url, headers=headers, stream=True, timeout=TIMEOUT_S) as response:
        response.raise_for_status()
        total = int(response.headers.get("Content-Length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc="Pleiades dump") as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
                bar.update(len(chunk))


download_pleiades(PLEIADES_URL, PLEIADES_GZ)


## Build the index

Parse the gzipped JSON `@graph` array into two dictionaries:

- `by_uri`: `uri -> compact place dict` (`uri, title, place_types, time_periods, lat, lon, all_names`)
- `by_name`: `normalized name -> [place dict, ...]`, indexing the title plus every `romanized` and `attested` name form

Latitude/longitude is taken from `reprPoint` when available, otherwise from the centroid of the first feature geometry. The pair is pickled to `PLEIADES_INDEX` for fast reuse.

In [ ]:
def feature_centroid(features: list[dict]) -> tuple[float, float] | None:
    """Derive an approximate (lon, lat) centroid from the first feature geometry."""
    for feature in features or []:
        geometry = feature.get("geometry") or {}
        coords = geometry.get("coordinates")
        if not coords:
            continue
        gtype = geometry.get("type")
        points = []
        if gtype == "Point":
            points = [coords]
        elif gtype in ("LineString", "MultiPoint"):
            points = coords
        elif gtype in ("Polygon", "MultiLineString"):
            points = [pt for ring in coords for pt in ring]
        elif gtype == "MultiPolygon":
            points = [pt for poly in coords for ring in poly for pt in ring]
        if points:
            lons = [p[0] for p in points]
            lats = [p[1] for p in points]
            return (sum(lons) / len(lons), sum(lats) / len(lats))
    return None


def build_pleiades_index(gz_path: Path) -> tuple[dict, dict]:
    """Parse the Pleiades dump and build the by_uri and by_name lookup indices."""
    by_uri: dict[str, dict] = {}
    by_name: dict[str, list[dict]] = {}

    logger.info("Parsing %s", gz_path)
    with gzip.open(gz_path, "rt", encoding="utf-8") as f:
        data = json.load(f)

    for place in tqdm(data["@graph"], desc="Indexing Pleiades places"):
        uri = place.get("uri")
        title = place.get("title", "")
        if not uri:
            continue

        names = place.get("names", []) or []
        all_names = [title] if title else []
        all_names += [n["romanized"] for n in names if n.get("romanized")]
        all_names += [n["attested"] for n in names if n.get("attested")]
        all_names = sorted(set(n for n in all_names if n))

        repr_point = place.get("reprPoint")
        if repr_point:
            lon, lat = repr_point[0], repr_point[1]
        else:
            centroid = feature_centroid(place.get("features", []))
            lon, lat = centroid if centroid else (None, None)

        place_dict = {
            "uri": uri,
            "title": title,
            "place_types": place.get("placeTypes", []),
            "time_periods": place.get("timePeriods", []),
            "lat": lat,
            "lon": lon,
            "all_names": all_names,
        }
        by_uri[uri] = place_dict

        for name in all_names:
            key = normalize(name)
            if key:
                by_name.setdefault(key, []).append(place_dict)

    return by_uri, by_name


if PLEIADES_INDEX.exists() and not FORCE_REBUILD:
    logger.info("Loading existing index from %s", PLEIADES_INDEX)
    with open(PLEIADES_INDEX, "rb") as f:
        by_uri, by_name = pickle.load(f)
else:
    by_uri, by_name = build_pleiades_index(PLEIADES_GZ)
    PLEIADES_INDEX.parent.mkdir(parents=True, exist_ok=True)
    with open(PLEIADES_INDEX, "wb") as f:
        pickle.dump((by_uri, by_name), f)
    logger.info("Saved index to %s", PLEIADES_INDEX)

print(f"Total places indexed:    {len(by_uri)}")
print(f"Total unique name keys:  {len(by_name)}")


### Example lookups

In [ ]:
for query in ["marzabotto", "veii", "rome", "carthage"]:
    matches = by_name.get(normalize(query), [])
    print(f"\n'{query}' -> {len(matches)} match(es)")
    for place in matches[:3]:
        print(f"  {place['title']!r}  {place['uri']}  types={place['place_types']}")


## GeoNames fallback

> **Note:** using the GeoNames API requires a free account registered at [geonames.org](https://www.geonames.org/login) with the web-services privilege enabled; set `GEONAMES_USER` to your username.

`search_geonames` queries the REST API, filters by feature class, caches results in memory to avoid duplicate calls, and is used **only** as a last-resort fallback for toponyms left unresolved after all three Pleiades passes in `04_disambiguation.ipynb` (where the function is redefined identically).

In [ ]:
_geonames_cache: dict[str, list[dict]] = {}


def search_geonames(toponym: str) -> list[dict]:
    """Query the GeoNames API for a toponym, returning up to GEONAMES_ROWS candidates."""
    if toponym in _geonames_cache:
        return _geonames_cache[toponym]

    params = {
        "q": toponym,
        "maxRows": GEONAMES_ROWS,
        "featureClass": GEONAMES_FCLS,
        "username": GEONAMES_USER,
    }
    headers = {"User-Agent": USER_AGENT}
    try:
        response = requests.get(GEONAMES_URL, params=params, headers=headers, timeout=TIMEOUT_S)
        response.raise_for_status()
        payload = response.json()
    except requests.exceptions.RequestException as exc:
        logger.error("GeoNames query failed for %r: %s", toponym, exc)
        _geonames_cache[toponym] = []
        return []

    results = [
        {
            "geonames_id": entry.get("geonameId"),
            "name": entry.get("name"),
            "lat": float(entry["lat"]) if entry.get("lat") else None,
            "lon": float(entry["lng"]) if entry.get("lng") else None,
            "feature_class": entry.get("fcl"),
            "feature_code": entry.get("fcode"),
            "country_code": entry.get("countryCode"),
            "admin1_name": entry.get("adminName1"),
        }
        for entry in payload.get("geonames", [])
    ]
    _geonames_cache[toponym] = results
    return results


### Test GeoNames lookups

In [ ]:
for query in ["Marzabotto", "Tiber", "Vesuvius", "Alesia"]:
    results = search_geonames(query)
    print(f"\n'{query}' -> {len(results)} result(s)")
    for r in results:
        print(f"  {r['name']!r} ({r['feature_class']}/{r['feature_code']}) "
              f"{r['country_code']} {r['admin1_name']!r}  lat={r['lat']} lon={r['lon']}  "
              f"id={r['geonames_id']}")
